# Phase 6 summarization: LoRA, LoRMA, full fine-tuning and distillation on my phase 1 gqa_swa model

Attach the phase 1 output (`gqa_swa_best.pt` and `bpe_legal_16k_v2.pkl`) and add the `HF` secret.

In [ ]:
import os, gc, re, json, math, time, pickle, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download

# install rouge-score if it is missing
try:
    from rouge_score import rouge_scorer
except ImportError:
    os.system("pip install -q rouge-score")
    from rouge_score import rouge_scorer


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for secret_name in ("HF_TOKEN", "HF"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(secret_name)
            if HF_TOKEN:
                break
        except Exception:
            pass
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("Logged in.")
else:
    print("WARNING: HF_TOKEN not found, public datasets will download without auth.")

## Config

In [ ]:
# the document is cut to MAX_CTX tokens (head + tail), the summary to MAX_SUM tokens
MAX_CTX = 768
MAX_SUM = 256
BATCH_SIZE = 4
EPOCHS = 4
LR = 3e-4
WEIGHT_DECAY = 0.01
SEED = 42

RANK, ALPHA = 8, 16.0
# full = the top FULL_TOP_LAYERS blocks plus final norm and lm head train, everything else is frozen
FULL_TOP_LAYERS = 4
KD_TEMP = 2.0
KD_WEIGHT = 0.5

# generation: greedy, every token already generated gets its logit divided by REP_PENALTY
MAX_NEW_TOKENS = 128
REP_PENALTY = 1.2
# the dev rouge after every epoch only looks at the first EVAL_DOCS dev cases, for speed
EVAL_DOCS = 20

EXPERIMENTS = [
    {"num": 1, "name": "lora", "method": "lora"},
    {"num": 2, "name": "lorma", "method": "lorma"},
    {"num": 3, "name": "full", "method": "full"},
    {"num": 4, "name": "distillation", "method": "distillation", "kd": True},
]

WORK_DIR = Path("/kaggle/working")
BASE_DIR = WORK_DIR / "phase6_summ"
BASE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if USE_AMP else "")

## Phase 1 checkpoint and tokenizer

In [ ]:
# first match under /kaggle/input
def resolve_artifact(filename):
    for root, _, files in os.walk("/kaggle/input"):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"Missing {filename}")


CKPT_PATH = resolve_artifact("gqa_swa_best.pt")
TOK_PATH = resolve_artifact("bpe_legal_16k_v2.pkl")
print("checkpoint:", CKPT_PATH)
print("tokenizer :", TOK_PATH)


class CustomBPETokenizer:
    def __init__(self, vocab_size=16384):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>"]
        self.pad_token_id, self.unk_token_id, self.bos_token_id, self.eos_token_id = 0, 1, 2, 3
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?\S|\s+(?!\S)|\s+""")
        self.merges, self.vocab, self.inverse_vocab, self._word_cache = {}, {}, {}, {}

    def _tokenize_word(self, word):
        word_tuple = tuple(word)
        if len(word_tuple) <= 1:
            return list(word_tuple)
        while True:
            pairs = [(word_tuple[i], word_tuple[i + 1]) for i in range(len(word_tuple) - 1)]
            valid = [p for p in pairs if p in self.merges]
            if not valid:
                break
            best = min(valid, key=lambda p: self.vocab[self.merges[p]])
            merged = self.merges[best]
            new_tuple, i = [], 0
            while i < len(word_tuple):
                if i < len(word_tuple) - 1 and word_tuple[i] == best[0] and word_tuple[i + 1] == best[1]:
                    new_tuple.append(merged)
                    i += 2
                else:
                    new_tuple.append(word_tuple[i])
                    i += 1
            word_tuple = tuple(new_tuple)
            if len(word_tuple) == 1:
                break
        return list(word_tuple)

    def encode(self, text, add_special_tokens=False):
        if not text:
            return []
        ids = []
        for word in self.pat.findall(text):
            if word not in self._word_cache:
                self._word_cache[word] = [self.vocab.get(t, self.unk_token_id) for t in self._tokenize_word(word)]
            ids.extend(self._word_cache[word])
        if add_special_tokens:
            ids = [self.bos_token_id] + ids + [self.eos_token_id]
        return ids

    # the first 4 ids are special tokens, they get dropped
    def decode(self, token_ids, skip_special_tokens=True):
        return "".join([self.inverse_vocab.get(t, "<unk>") for t in token_ids if not (skip_special_tokens and t < 4)])

    @classmethod
    def load(cls, filepath):
        with open(filepath, "rb") as f:
            data = pickle.load(f)
        tok = cls(data["target_vocab_size"])
        tok.vocab, tok.inverse_vocab, tok.merges = data["vocab"], data["inverse_vocab"], data["merges"]
        print(f"Loaded tokenizer from {filepath} (Vocab size: {len(tok.vocab)})")
        return tok


tokenizer = CustomBPETokenizer.load(TOK_PATH)

## Data

In [ ]:
REPO = "Exploration-Lab/CS-781-Capstone"
P6 = "Phase_6_Summ"


# the json can be a list, {"data": [...]} or a dict keyed by id, we always return a list of records
def fetch_json(rel_path):
    path = hf_hub_download(repo_id=REPO, filename=rel_path, repo_type="dataset", token=HF_TOKEN)
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "data" in data:
        data = data["data"]
    elif isinstance(data, dict):
        data = [{"id": k, **(v if isinstance(v, dict) else {"document": v})} for k, v in data.items()]
    return data


def get_document(rec):
    return rec.get("document", rec.get("text", ""))


train_raw = fetch_json(f"{P6}/Train/train.json")
test_data = fetch_json(f"{P6}/Test/test.json")

# there is no dev file, so 15% of train is held out
random.shuffle(train_raw)
split_idx = int(len(train_raw) * 0.85)
train_data = train_raw[:split_idx]
dev_data = train_raw[split_idx:]
print(f"train {len(train_data)} | dev {len(dev_data)} | test {len(test_data)}")

## Prompts and masking

In [ ]:
# every case becomes: <s> Document: [head + tail of the text] \n\nSummary: [summary] </s>
# the labels are -100 on the whole prompt, so the loss only covers the summary
class SummDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        rec = self.records[i]
        doc_ids = tokenizer.encode(get_document(rec))
        sum_ids = tokenizer.encode(rec.get("summary", ""))

        # long documents keep the first and the last half of MAX_CTX
        if len(doc_ids) > MAX_CTX:
            doc_ids = doc_ids[:MAX_CTX // 2] + doc_ids[-MAX_CTX // 2:]
        sum_ids = sum_ids[:MAX_SUM - 1] + [tokenizer.eos_token_id]

        prompt = [tokenizer.bos_token_id] + tokenizer.encode("Document: ", add_special_tokens=False) + doc_ids + tokenizer.encode("\n\nSummary: ", add_special_tokens=False)
        input_ids = prompt + sum_ids
        labels = [-100] * len(prompt) + sum_ids
        return torch.tensor(input_ids), torch.tensor(labels)


def collate_fn(batch):
    L = max(len(x[0]) for x in batch)
    # pad ids with the pad token and labels with -100
    ids = torch.full((len(batch), L), tokenizer.pad_token_id, dtype=torch.long)
    lbl = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, (inp, label) in enumerate(batch):
        ids[i, :len(inp)] = inp
        lbl[i, :len(label)] = label
    return ids.to(DEVICE), lbl.to(DEVICE)


train_loader = DataLoader(SummDataset(train_data), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(SummDataset(dev_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print("train batches", len(train_loader), "| dev batches", len(dev_loader))

## Phase 1 model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return self.weight * (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6))


class SwiGLU(nn.Module):
    def __init__(self, in_f, hid_f):
        super().__init__()
        self.w1 = nn.Linear(in_f, hid_f, bias=False)
        self.w2 = nn.Linear(in_f, hid_f, bias=False)
        self.w3 = nn.Linear(hid_f, in_f, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


def apply_rope(q, k):
    T, hd = q.size(-2), q.size(-1)
    # rotary position tables
    inv_freq = 1.0 / (10000 ** (torch.arange(0, hd, 2, device=q.device).float() / hd))
    t = torch.arange(T, device=q.device).float()
    emb = torch.cat([torch.outer(t, inv_freq)] * 2, dim=-1)
    cos, sin = emb.cos()[None, None, :], emb.sin()[None, None, :]

    def rot(x):
        return torch.cat((-x[..., hd // 2:], x[..., : hd // 2]), dim=-1)

    return q * cos + rot(q) * sin, k * cos + rot(k) * sin


# fewer k/v heads than q heads (grouped query)
class GQACausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, window=None):
        super().__init__()
        self.n_heads, self.n_kv_heads, self.head_dim = n_heads, n_kv_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        self.window = window

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k)
        k = k.repeat_interleave(self.n_heads // self.n_kv_heads, dim=1)
        v = v.repeat_interleave(self.n_heads // self.n_kv_heads, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y)


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, window=None):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQACausalAttention(d_model, n_heads, n_kv_heads, window)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.ffn(self.ffn_norm(x))


class LegalEfficientTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=1024, n_heads=16, n_kv_heads=4, n_layers=16, d_ff=2816, window=256):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        # every 4th layer is global, the others get the window
        self.layers = nn.ModuleList([
            EfficientDecoderBlock(d_model, n_heads, n_kv_heads, d_ff, None if (i + 1) % 4 == 0 else window)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx):
        x = self.tok_emb(idx)
        for layer in self.layers:
            # checkpoint the blocks while training, so it fits on the gpu
            x = grad_ckpt(layer, x, use_reentrant=False) if self.training else layer(x)
        return self.lm_head(self.final_norm(x))


ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
sd = {k.replace("_orig_mod.", "", 1): v for k, v in (ckpt.get("state_dict") or ckpt.get("model") or ckpt).items()}
cfg = ckpt.get("config", {"vocab_size": 16384, "d_model": 1024, "n_heads": 16, "n_kv_heads": 4, "n_layers": 16, "d_ff": 2816})


# a fresh copy of phase 1 every time, so experiments never share weights
def load_backbone():
    m = LegalEfficientTransformer(**cfg)
    m.load_state_dict(sd, strict=True)
    return m.to(DEVICE)


log("Phase 1 model and tokenizer ready.")

## Adapters

In [ ]:
# same idea as lora but the update acts on the output of W instead of the input
class LoRMALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16.0):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        out_f = base.weight.size(0)
        self.A = nn.Parameter(torch.randn(r, out_f) * 0.02)
        # B starts at zero so at the start it behaves exactly like phase 1
        self.B = nn.Parameter(torch.zeros(out_f, r))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * (F.dropout(h, 0.05) @ self.A.t() @ self.B.t())


# frozen W plus a small trainable B @ A
class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16.0):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, base.weight.size(1)) * 0.02)
        self.B = nn.Parameter(torch.zeros(base.weight.size(0), r))
        self.scale = alpha / r

    def forward(self, x):
        # W x + scale * B A x
        return self.base(x) + self.scale * (F.dropout(x, 0.05) @ self.A.t() @ self.B.t())


# wrap the target linears for the method (only the wrapped linears get frozen, the rest keeps requires_grad as loaded)
def make_model(method):
    m = load_backbone()

    if method == "lora":
        # attention projections and the swiglu mlp
        for b in m.layers:
            for n in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(b.attn, n, LoRALinear(getattr(b.attn, n), RANK, ALPHA))
            for n in ("w1", "w2", "w3"):
                setattr(b.ffn, n, LoRALinear(getattr(b.ffn, n), RANK, ALPHA))
            b.attn_norm.requires_grad_(True)
            b.ffn_norm.requires_grad_(True)
            m.final_norm.requires_grad_(True)

    elif method == "lorma":
        # attention projections only
        for b in m.layers:
            for n in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(b.attn, n, LoRMALinear(getattr(b.attn, n), RANK, ALPHA))
            b.attn_norm.requires_grad_(True)
            m.final_norm.requires_grad_(True)

    elif method == "full":
        # freeze everything, then open the top blocks, the final norm and the lm head (tied with the embedding)
        for p in m.parameters():
            p.requires_grad_(False)
        for b in m.layers[-FULL_TOP_LAYERS:]:
            b.requires_grad_(True)
        m.final_norm.requires_grad_(True)
        m.lm_head.requires_grad_(True)

    elif method == "distillation":
        # small student: every second phase 1 layer (8 of 16), lora on the attention projections
        m.layers = nn.ModuleList([m.layers[i] for i in range(0, len(m.layers), 2)])
        for b in m.layers:
            for n in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(b.attn, n, LoRALinear(getattr(b.attn, n), RANK, ALPHA))

    else:
        raise ValueError(f"unknown method: {method}")

    m = m.to(DEVICE)
    tr_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    tot_params = sum(p.numel() for p in m.parameters())
    print(f"[{method.upper()}] trainable {tr_params / 1e6:.2f}M / total {tot_params / 1e6:.2f}M ({tr_params / tot_params:.1%})")
    return m

## Generation, evaluation and submission

In [ ]:
# greedy decoding with a repetition penalty, the model reads the same prompt as in training
@torch.no_grad()
def generate_summary(model, text, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    ids = tokenizer.encode(text)
    if len(ids) > MAX_CTX:
        ids = ids[:MAX_CTX // 2] + ids[-MAX_CTX // 2:]
    prompt = [tokenizer.bos_token_id] + tokenizer.encode("Document: ", add_special_tokens=False) + ids + tokenizer.encode("\n\nSummary: ", add_special_tokens=False)

    input_tensor = torch.tensor([prompt], device=DEVICE)
    generated = []

    for _ in range(max_new_tokens):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(input_tensor)
        next_logits = logits[0, -1, :]

        # repetition penalty
        for t in set(generated):
            next_logits[t] /= REP_PENALTY

        next_token = next_logits.argmax().item()
        if next_token == tokenizer.eos_token_id:
            break
        generated.append(next_token)
        input_tensor = torch.cat([input_tensor, torch.tensor([[next_token]], device=DEVICE)], dim=1)

    return tokenizer.decode(generated).strip()


# rouge-1 and rouge-L on the first EVAL_DOCS dev cases
def evaluate_dev(model):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    r1, rl = [], []
    for rec in tqdm(dev_data[:EVAL_DOCS], desc="Evaluating", leave=False):
        pred = generate_summary(model, get_document(rec))
        scores = scorer.score(rec.get("summary", ""), pred)
        r1.append(scores["rouge1"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)
    return {"rouge1": np.mean(r1), "rougeL": np.mean(rl)}


# one summary for every test case, zipped as prediction.json
def generate_submission(model, exp_num, method_name):
    preds = []
    for rec in tqdm(test_data, desc=f"Predicting exp {exp_num}"):
        preds.append({"id": rec["id"], "summary": generate_summary(model, get_document(rec))})

    pred_path = BASE_DIR / f"prediction_{method_name}.json"
    zip_name = WORK_DIR / f"submission_exp{exp_num}_{method_name}.zip"

    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(preds, f, indent=2)

    # prediction.json has to be at the root of the zip
    with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")

    print(f"wrote {zip_name} | {len(preds)} summaries")
    return zip_name

## Training loop

In [ ]:
# trains for a fixed number of epochs, prints dev rouge every epoch
# teacher given = distillation (hard ce plus the temperature kl against the teacher)
def run_training(model, opt, scaler, tag, epochs, teacher=None):
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        bar = tqdm(train_loader, desc=f"{tag} ep{ep}/{epochs}", leave=False)

        for ids, labels in bar:
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(ids)

                # next token prediction, so the logits and the labels are shifted by one
                shift_logits = logits[..., :-1, :].contiguous()
                shift_labels = labels[..., 1:].contiguous()
                loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

                if teacher is not None:
                    with torch.no_grad():
                        t_logits = teacher(ids)
                    kd_loss = F.kl_div(
                        F.log_softmax(shift_logits / KD_TEMP, dim=-1),
                        F.softmax(t_logits[..., :-1, :] / KD_TEMP, dim=-1),
                        reduction="batchmean",
                    ) * (KD_TEMP ** 2)
                    loss = KD_WEIGHT * loss + (1 - KD_WEIGHT) * kd_loss

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            train_loss += loss.item()
            bar.set_postfix(loss=f"{train_loss / (bar.n + 1):.4f}")

        metrics = evaluate_dev(model)
        print(f"  [{tag}] ep {ep}: loss {train_loss / len(train_loader):.4f} | dev ROUGE-1 {metrics['rouge1']:.4f} ROUGE-L {metrics['rougeL']:.4f} | {(time.time() - t0) / 60:.1f} min")

## Train

In [ ]:
teacher = None

for exp in EXPERIMENTS:
    print(f"\n===== exp {exp['num']}: {exp['name']} =====")

    model = make_model(exp["method"])
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    # distillation uses the full run (exp 3) as the teacher
    run_training(model, optimizer, scaler, exp["name"], EPOCHS, teacher=teacher if exp.get("kd") else None)
    generate_submission(model, exp["num"], exp["name"])

    if exp["method"] == "full":
        # keep it in memory as the teacher for exp 4
        teacher = model
        teacher.eval()
        del optimizer, scaler
    else:
        del model, optimizer, scaler
    gc.collect(); torch.cuda.empty_cache()

log("all experiments done.")